In [0]:
# Imports: all notebook dependencies
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, StringType
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, percentile_approx
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, DecisionTreeClassifier, GBTClassifier, NaiveBayes, MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import urllib.request
import tempfile
import os
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from functools import reduce


# Load Wine Dataset

This section loads the UCI wine dataset, assigns column names, and checks for rescued data and nulls. A schema is applied for proper types.

In [0]:
# Download and save to specified file path if not exists
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"
file_path = "/Volumes/dsa/raw/dsa_files/wine/wine.csv"
if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

In [0]:
column_names = [
    "CLASS", "ALCOHOL", "MALICACID", "ASH", "ALCALINITY_OF_ASH", "MAGNESIUM", "TOTAL_PHENOLS", "FLAVANOIDS", 
    "NONFLAVANOID_PHENOLS", "PROANTHOCYANINS", "COLOR_INTENSITY", "HUE", "DILUTED_WINES", "PROLINE"
]
schema = StructType([
    StructField("CLASS", IntegerType(), True),
    StructField("ALCOHOL", DoubleType(), True),
    StructField("MALICACID", DoubleType(), True),
    StructField("ASH", DoubleType(), True),
    StructField("ALCALINITY_OF_ASH", DoubleType(), True),
    StructField("MAGNESIUM", IntegerType(), True),
    StructField("TOTAL_PHENOLS", DoubleType(), True),
    StructField("FLAVANOIDS", DoubleType(), True),
    StructField("NONFLAVANOID_PHENOLS", DoubleType(), True),
    StructField("PROANTHOCYANINS", DoubleType(), True),
    StructField("COLOR_INTENSITY", DoubleType(), True),
    StructField("HUE", DoubleType(), True),
    StructField("DILUTED_WINES", DoubleType(), True),
    StructField("PROLINE", IntegerType(), True)
])
wine_df = spark.read.csv(file_path, schema=schema, header=False)

# Preview structure, check for nulls or rescued data
wine_df.show(5)
wine_df.printSchema()

# Outlier Removal with PySpark

All numeric features are winsorized using interquartile ranges (IQR) to limit the influence of extreme values. This improves downstream statistics, visualization, and modeling outcomes.

In [0]:
# Pure PySpark outlier removal (winsorizing) for numeric features
from pyspark.sql.functions import expr, col, percentile_approx

numeric_cols = [c for c in wine_df.columns if c != 'CLASS' and wine_df.schema[c].dataType.typeName() in ['double', 'integer']]
outlier_limits = {}
for col_name in numeric_cols:
    Q1 = wine_df.approxQuantile(col_name, [0.25], 0.01)[0]
    Q3 = wine_df.approxQuantile(col_name, [0.75], 0.01)[0]
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_limits[col_name] = (lower, upper)

# Apply winsorizing in PySpark for all numeric columns
win_exprs = [
    expr(f'CASE WHEN {c} < {outlier_limits[c][0]} THEN {outlier_limits[c][0]} WHEN {c} > {outlier_limits[c][1]} THEN {outlier_limits[c][1]} ELSE {c} END').alias(c)
    if c in outlier_limits else col(c)
    for c in wine_df.columns
]
wine_df = wine_df.select(*win_exprs)

# Exploratory Data Analysis (EDA)

The following cells cover null analysis, variance, outlier detection, class distribution, and basic statistics using pandas for convenient tabular summaries.

In [0]:
row_count = wine_df.count()
col_count = len(wine_df.columns)
print((row_count, col_count))

In [0]:
wine_df.describe().display()

In [0]:
# Convert to pandas and fix types
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pdf = wine_df.toPandas()
pdf['MAGNESIUM'] = pdf['MAGNESIUM'].astype(float)
pdf['PROLINE'] = pdf['PROLINE'].astype(float)

In [0]:
# Null analysis of wine_df
nulls = pd.DataFrame(pdf.isnull().sum(), columns=['null_count'])
display(nulls)

In [0]:
# Variance analysis of numeric columns
variance = pdf.var().sort_values(ascending=False)
display(pd.DataFrame(variance, columns=['variance']))

In [0]:
# Outlier summary using IQR method for each numeric feature
def outlier_summary(df):
    stats = {}
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            Q1, Q3 = df[c].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outlier_count = ((df[c] < lower) | (df[c] > upper)).sum()
            stats[c] = outlier_count
    return pd.DataFrame.from_dict(stats, orient='index', columns=['outlier_count'])
display(outlier_summary(pdf))

In [0]:
# Class distribution
class_counts = pdf['CLASS'].value_counts().sort_index()
display(pd.DataFrame(class_counts).rename(columns={'CLASS': 'count'}))

In [0]:
# Basic statistics preview
display(pdf.describe().T)

# Boxplots and Histograms

Feature distributions are visualized with boxplots and histograms for each numeric column. This helps you quickly spot outliers, skewness, and variance patterns.

In [0]:
# Boxplots and histograms by feature
plt.figure(figsize=(16, 10))
numeric_cols = [col for col in pdf.columns if col != 'CLASS' and pd.api.types.is_numeric_dtype(pdf[col])]
for i, col in enumerate(numeric_cols):
    plt.subplot(4, 4, i + 1)
    sns.boxplot(y=pdf[col])
    plt.title(f'Boxplot: {col}')
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 10))
for i, col in enumerate(numeric_cols):
    plt.subplot(4, 4, i + 1)
    sns.histplot(pdf[col], kde=True, bins=20)
    plt.title(f'Histogram: {col}')
plt.tight_layout()
plt.show()

# Grouped Class Visualizations

This section compares feature distributions across wine classes. Class-segmented boxplots and histograms illustrate how individual features vary between classes.

In [0]:
# Boxplots and histograms grouped by CLASS
plt.figure(figsize=(20, 12))
for i, col in enumerate(numeric_cols):
    plt.subplot(4, 4, i + 1)
    sns.boxplot(x=pdf['CLASS'], y=pdf[col], palette='Set2')
    plt.title(f'{col} by CLASS')
plt.tight_layout()
plt.show()

plt.figure(figsize=(20, 12))
for i, col in enumerate(numeric_cols):
    plt.subplot(4, 4, i + 1)
    for c in sorted(pdf['CLASS'].unique()):
        sns.kdeplot(pdf[pdf['CLASS'] == c][col], label=str(c), fill=True, linewidth=1)
    plt.title(f'Histogram: {col} by CLASS')
    if i == 0:
        plt.legend()
plt.tight_layout()
plt.show()

# Feature Engineering for ML

Features are assembled for machine learning using PySpark's VectorAssembler, creating a column suitable for model input.

In [0]:
# Set up MLflow experiment
mlflow.set_experiment("/Users/viniciusdvale@gmail.com/ml_wine_classification")

In [0]:
# Feature engineering
feature_cols = [c for c in wine_df.columns if c != "CLASS"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
wine_vec_df = assembler.transform(wine_df).select("features", "CLASS")

# Train/Test Split

Splits the dataset into training and test sets for unbiased evaluation. The split uses an 80/20 ratio with a fixed random seed.

In [0]:
# Split dataset
train_df, test_df = wine_vec_df.randomSplit([0.8, 0.2], seed=42)

# Class Balancing with Oversampling

Applies random oversampling to balance minority wine classes in the training set, ensuring models train on equally represented classes.

In [0]:
# Balance the training set using random oversampling
def oversample_minority_classes(df, label_col):
    from pyspark.sql.functions import col
    from functools import reduce
    class_counts = df.groupBy(label_col).count().collect()
    max_count = max([row['count'] for row in class_counts])
    oversampled_dfs = []
    for row in class_counts:
        cls = row[label_col]
        count = row['count']
        ratio = int(max_count / count)
        remainder = max_count % count
        df_cls = df.filter(col(label_col) == cls)
        oversampled = df_cls
        for _ in range(ratio - 1):
            oversampled = oversampled.union(df_cls)
        if remainder > 0:
            oversampled = oversampled.union(df_cls.limit(remainder))
        oversampled_dfs.append(oversampled)
    return reduce(lambda a, b: a.union(b), oversampled_dfs)

# Assemble features for MLlib (if not already vectorized)
if 'features' not in train_df.columns:
    from pyspark.ml.feature import VectorAssembler
    feature_cols = [col for col in train_df.columns if col != "CLASS"]
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    train_df_vec = assembler.transform(train_df).select("features", "CLASS")
else:
    train_df_vec = train_df

train_df_balanced = oversample_minority_classes(train_df_vec, "CLASS")

# Check Class Balance

Displays the final class counts after oversampling to confirm each wine class is equally represented in the training data.

In [0]:
train_df_balanced.groupBy("CLASS").count().display()

# Standardize Features

Uses StandardScaler to scale features in both training and test sets, enabling fair model training and performance evaluation.

In [0]:
# StandardScaler expects features named 'features' (already vectorized in train_df_balanced)
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model = scaler.fit(train_df_balanced)
train_df_balanced_scaled = scaler_model.transform(train_df_balanced)

# Prepare test features for scaling (already vectorized as test_df)
test_df_scaled = scaler_model.transform(test_df)

# Modeling and MLflow Logging

This section trains multiple classifiers, logs key metrics and confusion matrices in MLflow for each model, allowing robust comparison and performance tracking.

In [0]:
# Model training and predictions
models = [
    LogisticRegression(featuresCol="scaled_features", labelCol="CLASS", maxIter=100),
    RandomForestClassifier(featuresCol="scaled_features", labelCol="CLASS"),
    DecisionTreeClassifier(featuresCol="scaled_features", labelCol="CLASS")
]
fitted_models = []
predictions_list = []
for model in models:
    fitted = model.fit(train_df_balanced_scaled)
    predictions = fitted.transform(test_df_scaled)
    fitted_models.append(fitted)
    predictions_list.append(predictions)

In [0]:
# Metric calculation and MLflow logging
import mlflow
metric_results = []
for model, predictions in zip(models, predictions_list):
    if mlflow.active_run():
        mlflow.end_run()
    with mlflow.start_run(run_name=model.__class__.__name__):
        evaluator = MulticlassClassificationEvaluator(labelCol="CLASS", predictionCol="prediction")
        accuracy = evaluator.setMetricName("accuracy").evaluate(predictions)
        f1 = evaluator.setMetricName("f1").evaluate(predictions)
        precision = evaluator.setMetricName("weightedPrecision").evaluate(predictions)
        recall = evaluator.setMetricName("weightedRecall").evaluate(predictions)
        metric_results.append({
            "model": model.__class__.__name__,
            "accuracy": accuracy,
            "f1": f1,
            "precision": precision,
            "recall": recall
        })
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)

In [0]:
# Confusion matrix calculation and MLflow logging
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import tempfile
import os
conf_matrix_figs = []
for model, predictions in zip(models, predictions_list):
    y_true = predictions.select("CLASS").toPandas().astype(int)
    y_pred = predictions.select("prediction").toPandas().astype(int)
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax)
    plt.title(f"Confusion Matrix: {model.__class__.__name__}")
    tmpdir = tempfile.mkdtemp()
    img_path = os.path.join(tmpdir, f"confusion_matrix_{model.__class__.__name__}.png")
    plt.savefig(img_path)
    mlflow.log_artifact(img_path)
    conf_matrix_figs.append(fig)
    plt.close(fig)

In [0]:
# Display summary results
import pandas as pd
display(pd.DataFrame(metric_results))

In [0]:
# Show confusion matrices for all models
from matplotlib import pyplot as plt
for fig in conf_matrix_figs:
    plt.figure(fig.number)
    plt.show()

In [0]:
# Model Tuning and Champion vs Contestant Selection
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [0]:
model_names = ["LogisticRegression", "RandomForestClassifier", "DecisionTreeClassifier"]
grids = []
# Parameter grids for each
# LogisticRegression tuning
paramGrid_lr = (ParamGridBuilder()
    .addGrid(models[0].regParam, [0.0, 0.01, 0.1])
    .addGrid(models[0].elasticNetParam, [0.0, 0.5, 1.0])
    .build())
# RandomForest tuning
paramGrid_rf = (ParamGridBuilder()
    .addGrid(models[1].numTrees, [10, 30, 50])
    .addGrid(models[1].maxDepth, [5, 10, 20])
    .build())
# DecisionTree tuning
paramGrid_dt = (ParamGridBuilder()
    .addGrid(models[2].maxDepth, [2, 5, 10])
    .addGrid(models[2].minInstancesPerNode, [1, 5])
    .build())
grids = [paramGrid_lr, paramGrid_rf, paramGrid_dt]

evaluator = MulticlassClassificationEvaluator(labelCol="CLASS", predictionCol="prediction", metricName="accuracy")

best_models = []
tuning_results = []

In [0]:
import os
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/dsa/model/vol_mlflow_experiments'
print("Set SPARKML_TEMP_DFS_PATH to /Volumes/dsa/model/vol_mlflow_experiments for MLlib tuning support.")

In [0]:
for model, grid, name in zip(models, grids, model_names):
    if mlflow.active_run():
        mlflow.end_run()
    with mlflow.start_run(run_name=f"tune_{name}"):
        cv = CrossValidator(estimator=model, estimatorParamMaps=grid, evaluator=evaluator, numFolds=5)
        cvModel = cv.fit(train_df_balanced_scaled)
        predictions = cvModel.transform(test_df_scaled)
        accuracy = evaluator.evaluate(predictions)
        best_models.append(cvModel.bestModel)
        # FIX: Use extractParamMap for param extraction
        params = {param.name: cvModel.bestModel.extractParamMap().get(param) for param in grid[0].keys()}
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", accuracy)
        tuning_results.append({"model": name, "accuracy": accuracy, **params})
        y_true = predictions.select("CLASS").toPandas().astype(int)
        y_pred = predictions.select("prediction").toPandas().astype(int)
        from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
        import matplotlib.pyplot as plt
        import tempfile, os
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        fig, ax = plt.subplots(figsize=(6, 6))
        disp.plot(ax=ax)
        plt.title(f"Confusion Matrix: {name} (tuned)")
        tmpdir = tempfile.mkdtemp()
        img_path = os.path.join(tmpdir, f"cm_{name}.png")
        plt.savefig(img_path)
        mlflow.log_artifact(img_path)
        plt.show()

champion = max(tuning_results, key=lambda x: x["accuracy"])
tuning_df = pd.DataFrame(tuning_results)
display(tuning_df.sort_values(by="accuracy", ascending=False))
print(f"Champion model: {champion['model']} with accuracy={champion['accuracy']}\nParameters: { {k:v for k, v in champion.items() if k not in ['model', 'accuracy']} }")

In [0]:
# Predict CLASS with the champion model from best tuning

# New sample data
new_data = [
    [13.72, 1.43, 2.5, 16.7, 108, 3.4, 3.67, 0.19, 2.04, 6.8, 0.89, 2.87, 1285],
    [12.37, 0.94, 1.36, 10.6, 88, 1.98, 0.57, 0.28, 0.42, 1.95, 1.05, 1.82, 520],
    [11.27, 0.84, 2.18, 17.8, 67, 2.34, 0.49, 0.39, 0.71, 4.41, 1.12, 1.90, 765]
]
feature_names = [
    "ALCOHOL", "MALICACID", "ASH", "ALCALINITY_OF_ASH", "MAGNESIUM", "TOTAL_PHENOLS", "FLAVANOIDS",
    "NONFLAVANOID_PHENOLS", "PROANTHOCYANINS", "COLOR_INTENSITY", "HUE", "DILUTED_WINES", "PROLINE"
]

# Create Spark DataFrame
new_df = spark.createDataFrame(new_data, feature_names)

# Assemble features
assembler = VectorAssembler(inputCols=feature_names, outputCol="features")
new_vec_df = assembler.transform(new_df)

# Scale features using previously fitted scaler_model
new_scaled_df = scaler_model.transform(new_vec_df)

# Load champion model
champion_model = best_models[model_names.index(champion['model'])]

# Predict CLASS
predictions = champion_model.transform(new_scaled_df)
preds_pd = predictions.select("prediction").toPandas().astype(int)
display(preds_pd.rename(columns={"prediction": "Predicted CLASS"}))